# MovieLens Scala pipeline: evidence and bounded EDA
This notebook inspects actual Scala artifacts. It does not download, install, launch Spark, train models or load the 32M ratings into Python. Outputs are empty until explicitly executed. Spark compilation/runtime remain required acceptance checks.

In [ ]:
from pathlib import Path
import json
WORKSPACE = Path.cwd().resolve()
if WORKSPACE.name == 'notebooks':
    WORKSPACE = WORKSPACE.parent
assert (WORKSPACE / 'ARCHITECTURE.md').is_file()
RUN_DIRECTORY = None  # Set to an actual completed data/processed/spark/<run> directory.
READ_SMALL_GOLD_TABLES = False  # Requires manually installed pyarrow.


## Read measured run reports
The reports must come from the Scala job. A missing report means the job has not produced that evidence. Counts and timings are never supplied as notebook constants.

In [ ]:
reports = {}
if RUN_DIRECTORY is None:
    print('No Spark run selected. Follow documentation/bda/RUNBOOK.md first.')
else:
    run = (WORKSPACE / RUN_DIRECTORY).resolve()
    assert run.is_relative_to(WORKSPACE / 'data/processed')
    for path in sorted((run / 'reports').glob('*.json')):
        assert path.stat().st_size <= 2 * 1024 * 1024
        reports[path.name] = json.loads(path.read_text(encoding='utf-8'))
    print(json.dumps(reports, indent=2))


## Bounded aggregate inspection
Read only small aggregated Gold tables. Every view uses a row limit. Star rating statistics, review sentiment and recommendations remain separate. No code here reads raw or Silver ratings into pandas. For plots, export only these small aggregate views after manual plotting-dependency setup.

In [ ]:
if READ_SMALL_GOLD_TABLES:
    assert RUN_DIRECTORY is not None, 'Choose an actual completed run.'
    import pyarrow.dataset as ds
    for name in ('rating_distribution', 'rating_activity_monthly', 'genre_rating_statistics'):
        target = run / 'gold' / name
        if target.is_dir():
            view = ds.dataset(target, format='parquet').head(500)
            print(name, view.to_pylist())
        else:
            print(name, 'not produced')


## Academic evidence checklist
Inspect the saved RDD DAG, typed Dataset and join plans, reconciliation reports and Spark event logs. Capture the actual localhost:4040 UI while a manually launched job is active. Compare first-action and cached timings without assuming caching is faster. Discuss missing genres, the relationship quarantine, tag-only item cold starts and multi-genre denominators. No Spark metrics have been measured merely by creating this notebook.